In [ ]:
# GitHub token for private clone
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("AIC2026-PACs-token")


In [ ]:
# Fresh repo checkout in /kaggle/working
!rm -rf /kaggle/working/AIC2026-PACs
!git clone https://{secret_value_0}@github.com/AkiyaNguyen/AIC2026-PACs.git


In [ ]:
# Base deps include sentence-transformers (ASR segment embed)
!pip install -q -r /kaggle/working/AIC2026-PACs/requirements.txt


In [ ]:
# Edit: folders that contain (or nest) VIDEO_ID.jsonl from asr.zip unpack.
# Notebook rglob("*.jsonl") under each path.
ASR_DIRS = [
    "/kaggle/input/datasets/akiyanguyen/pacs-asr-l21",
    # "/kaggle/input/datasets/akiyanguyen/pacs-asr-l22",
    # "/kaggle/input/datasets/akiyanguyen/pacs-asr-l23",
    # "/kaggle/input/datasets/akiyanguyen/pacs-asr-l24",
    # "/kaggle/input/datasets/akiyanguyen/pacs-asr-l25",
    # "/kaggle/input/datasets/akiyanguyen/pacs-asr-l26",
    # "/kaggle/input/datasets/akiyanguyen/pacs-asr-l27",
    # "/kaggle/input/datasets/akiyanguyen/pacs-asr-l28",
    # "/kaggle/input/datasets/akiyanguyen/pacs-asr-l29",
    # "/kaggle/input/datasets/akiyanguyen/pacs-asr-l30",
]


In [ ]:
# Output mirrors clip layout: features/asr_emb/Lxx/VIDEO_ID.{npy,jsonl}
import json
import re
from pathlib import Path

REPO = "/kaggle/working/AIC2026-PACs"
OUT_ROOT = "/kaggle/working/features/asr_emb"
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
DEVICE = "cuda"  # cuda | cpu
BATCH_SIZE = 64
BATCH_RE = re.compile(r"^(L\d+)_", re.I)

missing = [d for d in ASR_DIRS if not Path(d).is_dir()]
if missing:
    raise SystemExit(f"Not a directory: {missing}")

jsonl_files = []
seen = set()
for d in ASR_DIRS:
    for p in sorted(Path(d).rglob("*.jsonl")):
        key = p.stem
        if key in seen:
            print(f"warning: duplicate stem {key}, keep first", flush=True)
            continue
        seen.add(key)
        jsonl_files.append(p)

if not jsonl_files:
    raise SystemExit(f"No .jsonl under ASR_DIRS={ASR_DIRS}")

print(f"jsonl_count={len(jsonl_files)}")
print("OUT_ROOT =", OUT_ROOT)
print("MODEL_NAME =", MODEL_NAME)
print("DEVICE =", DEVICE)


In [ ]:
# Segment embed: skip empty text; write filtered jsonl + L2-normalized .npy (row i ↔ line i).
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

out_root = Path(OUT_ROOT)
out_root.mkdir(parents=True, exist_ok=True)

device = DEVICE if (DEVICE == "cpu" or torch.cuda.is_available()) else "cpu"
if DEVICE == "cuda" and device == "cpu":
    print("warning: CUDA requested but unavailable; using cpu", flush=True)

model = SentenceTransformer(MODEL_NAME, device=device)
dim = int(model.get_sentence_embedding_dimension())
print(f"model loaded dim={dim} device={device}", flush=True)


def load_segments(path: Path) -> list[dict]:
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)
        text = str(obj.get("text") or "").strip()
        if not text:
            continue
        rows.append(
            {
                "start": float(obj["start"]),
                "end": float(obj["end"]),
                "text": text,
            }
        )
    return rows


def batch_id(stem: str) -> str:
    m = BATCH_RE.match(stem)
    return m.group(1) if m else "unknown"


n_ok = 0
n_empty = 0
for i, src in enumerate(jsonl_files, start=1):
    stem = src.stem
    batch = batch_id(stem)
    dest_dir = out_root / batch
    dest_dir.mkdir(parents=True, exist_ok=True)
    npy_path = dest_dir / f"{stem}.npy"
    jsonl_path = dest_dir / f"{stem}.jsonl"

    segs = load_segments(src)
    if not segs:
        n_empty += 1
        print(f"[{i}/{len(jsonl_files)}] skip empty {stem}", flush=True)
        continue

    texts = [s["text"] for s in segs]
    emb = model.encode(
        texts,
        batch_size=BATCH_SIZE,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).astype(np.float32, copy=False)

    if emb.ndim != 2 or emb.shape[0] != len(segs):
        raise SystemExit(f"Bad embed shape {emb.shape} for {stem} n={len(segs)}")

    np.save(npy_path, emb)
    with jsonl_path.open("w", encoding="utf-8") as f:
        for s in segs:
            f.write(json.dumps(s, ensure_ascii=False) + "\n")

    n_ok += 1
    if i == 1 or i % 10 == 0 or i == len(jsonl_files):
        print(
            f"[{i}/{len(jsonl_files)}] {batch}/{stem}  segs={len(segs)}  → {npy_path}",
            flush=True,
        )

meta = {"model": MODEL_NAME, "dim": dim, "normalize": True}
(out_root / "model.json").write_text(
    json.dumps(meta, indent=2) + "\n", encoding="utf-8"
)
print(f"Done: wrote={n_ok}  empty_skipped={n_empty}  dim={dim}")
print("model.json =", out_root / "model.json")


In [ ]:
# Zip asr_emb/ (Lxx/*.npy + Lxx/*.jsonl + model.json). Unpack into features/asr_emb/.
import zipfile
from IPython.display import FileLink, display

root = Path(OUT_ROOT)
files = [p for p in root.rglob("*") if p.is_file()]
out = Path("/kaggle/working/asr_emb.zip")
out.unlink(missing_ok=True)
with zipfile.ZipFile(out, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1) as zf:
    for f in files:
        zf.write(f, arcname=str(f.relative_to(root)))
print(f"Wrote {out.name}: {out.stat().st_size/1e6:.1f} MB  ({len(files)} files)")
display(FileLink(str(out)))
